# Practical Core — 6 Indicators

| Aspect                          | Includes                                                                      |
| ------------------------------- | ----------------------------------------------------------------------------- |
| Monetary & Financial Conditions | Interest rates, central banks, liquidity, credit, stock/bond markets, banking |
| Inflation & Prices              | CPI, PPI, wages, commodity-driven price pressures                             |
| Real Economic Activity          | GDP, recession, manufacturing, services, investment, productivity             |
| Labor & Consumption             | Employment, wages, retail sales, consumer spending/confidence                 |
| Fiscal & Government             | Taxes, deficits, spending, regulation, stimulus                               |
| External Sector                 | Trade, exports/imports, FX, tariffs, current account                          |



In [ ]:
!pip install snorkel # Run once

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 3.0 MB/s eta 0:00:00


In [ ]:
import re
from snorkel.preprocess import preprocessor
from snorkel.labeling import labeling_function, PandasLFApplier, LFAnalysis
from snorkel.labeling.lf.nlp import nlp_labeling_function
from snorkel.labeling.model import LabelModel
from sentence_transformers import SentenceTransformer, util
from typing import List, Dict
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import WordNetLemmatizer
import numpy as np

In [ ]:
# Download required NLTK resources (run once)
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
# Embedder model instantiation
#embedder = SentenceTransformer('all-mpnet-base-v2')
#embedder = SentenceTransformer('mixedbread-ai/mxbai-embed-large-v1')
embedder = SentenceTransformer('sentence-transformers/all-distilroberta-v1')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import json

# Load JSON file
with open("predictions_review.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Flatten into rows
rows = []

for aspect_combo, samples in data.items():
    for sample in samples:
        rows.append({
            "aspects": aspect_combo,
            "article_id": sample.get("article_id"),
            "title": sample.get("title"),
            "sentence_idx": sample.get("sentence_idx"),
            "text": sample.get("text")
        })

# Create DataFrame
df = pd.DataFrame(rows)

# Optional: inspect
print(df.head())

                               aspects  article_id  \
0                                 None         164   
1                                 None           3   
2                      External_Sector          97   
3                      External_Sector          65   
4  External_Sector + Fiscal_Government         159   

                                               title  sentence_idx  \
0         Where experience meets academic excellence            14   
1  ECB urges banks to brace quickly for AI-assist...             8   
2  China's durian boom is ripening into a regiona...            21   
3  Johor eyeing 12 million visitors, RM42.48bil i...             7   
4              Cautious optimism on growth prospects             4   

                                                text  
0  “I needed a university that was recognised, ap...  
1  (Reporting by Ludwig Burger and Reinhard Becke...  
2  As Southeast Asia's main producing areas enter...  
3  He added that Singapore remains

In [ ]:
df.to_csv("benchmarks.csv", index=False)

In [ ]:
from nltk.util import pr
# Sentence anchors

SEMANTIC_THRESHOLD = 0.35 # THRESHOLD FOR SEMANTIC SIMILARITY

@preprocessor(memoize=True)
def cache_embeddings(x):
    if not hasattr(x, "embedding"):
        x.embedding = embedder.encode(x.text, convert_to_tensor=True)
    return x

# Economic news anchor sentences
ECONOMIC_ANCHORS = [
    "Gold seen bearish, according to global investor sentiments as reported by central bank",
    "A high-ranking government minister faces official questioning and institutional scrutiny regarding state-backed investment deals, public venture funds, or corporate partnerships with multinational tech firms.",
    "Government entities and ministries face regulatory oversight and institutional scrutiny regarding public capital allocations for strategic foreign direct investments, sovereign tech venture funding, and semiconductor supply chain partnerships.",
    "Analysts warned that prolonged inflation could weigh further on economic recovery prospects.",
    "Sovereign nations utilizing international emergency credit lines must settle outstanding central banking obligations and institutional stabilization loans to restore external financial credibility.",
    "Financial media outlets and international institutions provide regular analytical reporting on global macroeconomic trends, national development milestones, and long-term fiscal performance.",
    "The central bank changed interest rates and job market policy.",
    "A looming sovereign default, national debt crisis, or balance-of-payments collapse forces governments to seek emergency international bailouts and strict fiscal restructuring packages.",
    "The finance ministry announced additional subsidies to offset rising fuel and food prices.",
    "Trade activity improved in April, resulting in an increase in exports.",
    "Government spending adjustments, fiscal restraint, and national budget revisions.",
    "The stock market grew 5 points this quarter, with TES (Tesla) being the biggest winner",
    "Global oil prices surged past key psychological thresholds due to energy market disruptions.",
    "Supply chain bottlenecks and commodity shortages rattled global trade networks.",
    "The US dollar (USD) gained value against the Malaysian Ringgit (MYR)",
    "National employment statistics tracking own-account workers, workforce participation rates, and monthly changes in headcount populations.",
    "The government formalized its alignment with emerging market trade blocs, aiming to diversify international commerce networks and enhance regional investment cooperation.",
    "The Prime Minister defended the nation's economic track record, affirming that robust institutional policy frameworks would keep the investment ecosystem stable despite regional volatility.",
    "The international monetary fund updated its macroeconomic outlook for developing markets, citing mixed structural adjustments.",
    "The government has announced a multi-million ringgit financial allocation to fund new economic and national talent development initiatives."
]

# Aspect sentence anchors
# Monetary Policy
MONETARY_ANCHORS = [
    "central bank raised interest rates",
    "benchmark interest rate hike",
    "Sovereign nations utilizing international emergency credit lines must settle outstanding central banking obligations and institutional stabilization loans to restore external financial credibility.",
    "liquidity tightening measures",
    "The monetary policy committee signaled a potential reduction in reserve requirements to stimulate credit expansion."
]
# Inflation
INFLATION_ANCHORS = [
    "cost of living has gone up",
    "As skyrocketing diesel costs ripple through the supply chain, economists warn that unmitigated fuel hikes are stoking broader inflation, leaving both commercial enterprises and household consumers vulnerable to a sharp increase in the cost of living.",
    "prices continue to climb year-on-year",
    "inflationary pressures remain elevated",
    "Producer price indices ticked upward, indicating that rising raw material costs are beginning to pass through to wholesale market pricing and end-consumer goods."
]
# Economic Growth : GDP CPI Recession
ECONOMIC_GROWTH_ANCHORS = [
    "recession fears are mounting",
    "economy grew faster than expected",
    "the economy contracted sharply",
    "National policymakers must implement long-term structural reforms and fiscal adjustments to sustain macroeconomic growth and ensure future economic stability.",
    "Surging manufacturing output and strong industrial production cycles are expected to significantly boost quarterly economic growth."
    ]
# Labor Market : Unemployment rate, minimum wage
LABOR_ANCHORS = [
    "layoffs increased amid cost cuts",
    "labor market remains tight",
    "National employment statistics tracking own-account workers, workforce participation rates, and monthly changes in headcount populations.",
    "Workforce training programs, social security protections, human capital development, and gig worker welfare.",
    "Labor unions accused management of wage theft, unfair contract breaches, and retaliatory terminations during the worker dispute.",
    "The department of statistics reported that the seasonally adjusted unemployment rate held steady as payroll additions balanced out market entries."
]
# Consumer Activity: demands,
CONSUMER_ANCHORS = [
    "household demand remains resilient",
    "more people are buying goods",
    "retail sales rose strongly this month",
    'Private consumption expenditures showed robust quarter-on-quarter acceleration, outstripping conservative retail forecasts'
]
 # Business Activity: Investments
BUSINESS_ANCHORS = [
    "A sharp contraction or operational slowdown in major industrial sectors, including real estate, construction, and infrastructure, heavily disrupts business activity, project completions, and contract worker utilization.",
    "manufacturing activity expanded this month",
    "A systemic collapse or widespread bankruptcy of small and medium enterprises (SMEs) poses a severe threat to the national business environment and industrial production infrastructure.",
    "Domestic enterprises signaled increased capital allocation toward building new distribution centers and scaling local production facilities.",
    "ceo reports increased profits",
    "A high-ranking government minister faces official questioning and institutional scrutiny regarding state-backed investment deals, public venture funds, or corporate partnerships with multinational tech firms.",
    "Industrial manufacturing plants and corporate entities are modifying their core operating models, production schedules, and factory inventory practices."
]
# Financial Markets: Stocks, Bonds, Instruments
FINANCIAL_ANCHORS = [
    "stock markets rallied on strong earnings",
    "investors have bullish expectations",
    "google report record high profits, 3 points",
    "The S&P 500 shed 1.24% to end at 7,408.50, while the Nasdaq Composite slipped 1.54% to 26,225.14.",
    "At the close of trade, the stock index finished points higher.",
    "The benchmark index slipped down percentage points following intraday trading.",
    "Sovereign bond yields curve shifted as equity markets faced selloffs.",
    "The local currency opened higher against the US dollar ahead of market closing."
]
# Trade : Imports Exports
TRADE_ANCHORS = [
    "palm oil exports increased",
    "The Malaysian ringgit fluctuated in volatile trading, gaining ground against the Singapore dollar and edging up against the US dollar in the regional currency market.",
    "The domestic currency experienced cross-border valuation shifts, strengthening or weakening against major regional trading partner exchange rates in the global FX spot market.",
    "asia increased imports",
    "trade war between them continues",
    'Cross-border shipments rebounded sharply, narrowing the merchandise trade deficit as bilateral commerce normalized'
]
 # Fiscal Policy: Government spending and budget
FISCAL_ANCHORS = [
    # --- Government Spending & Subsidies ---
    "The finance ministry announced a major restructuring of targeted fuel and electricity subsidies.",
    "Government economic resilience packages, financial stimulus programs, and national emergency funding initiatives.",
    "The Prime Minister announced a national regulation plan to ensure the security  of strategic commodity and energy supplies.",
    "Government leadership emphasized that maintaining national economic growth requires difficult policy decisions regarding subsidy re-evaluation and fiscal restructuring.",
    # --- Taxation & Revenue Generation ---
    "Tax cuts introduced this year aim to boost disposable income and corporate reinvestment rates.",
    "Parliament debated raising corporate windfall taxes and expanding the scope of the consumption tax.",
    "The inland revenue board stepped up enforcement to curb tax evasion and broaden the national revenue base.",

    # --- Budgetary Frameworks & Deficits ---
    "The government outlined its revised budgetary framework, targeting a reduction in the fiscal deficit through streamlined public expenditures.",
    "The annual budget allocation prioritized healthcare, education, and rural development civil projects.",

    # --- Sovereign Debt & Fiscal Discipline ---
    "Economists warned that rising statutory debt levels could prompt international rating agencies to reassess the sovereign outlook.",
    "The treasury tap-issued new government investment issues (MGS/GII) to fund the fiscal deficit.",

    "A high-ranking government minister faces official questioning and institutional scrutiny regarding state-backed investment deals, public venture funds, or corporate partnerships with multinational tech firms."
]
 # Energy & Commodities; Oil, Solar, Energy, Metal
ENERGY_COMMODITIES_ANCHORS = [
    "Brazilian soy exports are projected to surge next month, boosting global port activity.",
    "As global coal mining hubs enter peak production season, maritime commodity imports are expected to climb sharply throughout the second quarter."
    "natural gas prices spiked during winter demand",
    "electricity and household utility bills",
    "Industrial metals like copper and iron ore faced downward price pressure as global warehouse inventories reached multi-month highs"
]
# Banking and Credit: Loan rates, mortgage, credit
BANKING_CREDIT_ANCHORS = [
    "household credit expanded steadily",
    "banks increased provisions for bad loans",
    "Commercial banks tightened their credit underwriting standards, leading to a marginal slowdown in corporate loan approvals."
]
# Corporate & Investment Climate:
CORPORATE_CLIMATE_ANCHORS = [
    "Inbound foreign direct investment surged as international conglomerates finalized cross-border capital allocations for regional infrastructure projects.",
    "firms expanded operations into new markets",
    "capital expenditure plans were delayed"
]

In [ ]:
# --- Build dictionary of string anchors (optional, keep for reference) ---
ANCHOR_STRINGS: Dict[str, List[str]] = {
    "monetary_policy": MONETARY_ANCHORS,
    "inflation": INFLATION_ANCHORS,
    "economic_growth": ECONOMIC_GROWTH_ANCHORS,
    "labor_market": LABOR_ANCHORS,
    "consumer_activity": CONSUMER_ANCHORS,
    "business_activity": BUSINESS_ANCHORS,
    "financial_markets": FINANCIAL_ANCHORS,
    "trade_external": TRADE_ANCHORS,
    "fiscal_policy": FISCAL_ANCHORS,
    "energy_commodities": ENERGY_COMMODITIES_ANCHORS,
    "banking_credit": BANKING_CREDIT_ANCHORS,
    "corporate_climate": CORPORATE_CLIMATE_ANCHORS
}

In [ ]:
# --- Convert to dict of list of tensors ---
ANCHORS: Dict[str, List] = {}  # Value type will be list[tensor]
for category, phrases in ANCHOR_STRINGS.items():
    ANCHORS[category] = [
        embedder.encode(phrase, convert_to_tensor=True) for phrase in phrases
    ]
economic_anchor_embeddings = []
for anchor in ECONOMIC_ANCHORS:
    economic_anchor_embeddings.append(embedder.encode(anchor, convert_to_tensor=True))
# Quick check: print shapes
for cat, emb_list in ANCHORS.items():
    print(f"{cat}: {len(emb_list)} anchors, each embedding shape {emb_list[0].shape}")

monetary_policy: 5 anchors, each embedding shape torch.Size([768])
inflation: 5 anchors, each embedding shape torch.Size([768])
economic_growth: 5 anchors, each embedding shape torch.Size([768])
labor_market: 6 anchors, each embedding shape torch.Size([768])
consumer_activity: 4 anchors, each embedding shape torch.Size([768])
business_activity: 7 anchors, each embedding shape torch.Size([768])
financial_markets: 8 anchors, each embedding shape torch.Size([768])
trade_external: 6 anchors, each embedding shape torch.Size([768])
fiscal_policy: 12 anchors, each embedding shape torch.Size([768])
energy_commodities: 4 anchors, each embedding shape torch.Size([768])
banking_credit: 3 anchors, each embedding shape torch.Size([768])
corporate_climate: 3 anchors, each embedding shape torch.Size([768])


In [ ]:
# Keywords for matching
KEYWORDS: dict[str, list[str]] = {
    "monetary_policy": [
        "fed", "central bank", "interest rate", "rate cut", "rate hike",
        "monetary", "bnm", "overnight policy rate", "opr"
    ],
    "inflation": [
        "inflation", "cpi", "ppi", "price pressure", "cost of living", "consumer price", 'price', 'prices', 'domestic prices'
    ],
    "economic_growth": [
        "gdp", "recession", "economic growth", "economic expansion",
         "maintain growth", "sustain growth", "gdp growth", "growth trajectory"
        "productivity gain", "output growth", "quarterly growth", "annual growth"
    ],
    "labor_market": [
        "unemployment", "layoff", "job market", "labor participation", "wage",
        "employment", "labor", "hiring", "job creation", "minimum wage", "immigration",
        "hire", "layoff", "downsize", "recruit", "fire", "terminate","gig worker", "social security",
        "skills development", "training", "workforce", 'worker', 'employee', 'headcount'
    ],
    "consumer_activity": [
        "household debt", "credit card usage", "disposable income", "luxury good",
        "sale", "retail sale", "spending", "consumer confidence", "consumer sentiment",
        "consumer spending", "household spending", "consumer demand"
    ],
   "business_activity": [
    "manufacturing", "service", "pmis", "industrial output", "ceo", "automation",
    "corporation", "sme", "enterprise", "firm", "conglomerate", "multinational",
    "private sector", "commercial sector", "industrial base", "ecosystem","revenue", "profit", "earning", "profitability",
    "yield", "output", "capacity utilization", "business", "turnover", "margin",
    "investment", "business investment", "business expansion", "capex",
    "capital expenditure", "asset acquisition", "facility", "infrastructure rollout",
    "plant", "equipment", "r&d", "research and development", "property right", "robust", "expansion",
    "slowdown", "growth", "disruption", "supply chain", "supply network",
    "collapse", "bankruptcy", "insolvency", "default", "liquidation",
    "restructuring", "distress", "closure", "red tape",  "contraction", "resilient", "slump", "stagnation",
    ],
    "financial_markets": [
        "stock", "bond", "bursa", "equity", "yield", "index",
        "bourse", "indices", "klci", "fbmklci", "derivative", "derivatives", "futures market", "etf", "exchange-traded fund", "reit", "sukuk",
        "liquidation", "sell-off", "rally", "bull market", "bear market", "market capitalization", "market cap", "trading volume", "turnover value", "equities market",
        "ipo", "initial public offering", "treasury bills", "t-bills", "capital markets", "debt securities", "share price", "dividend yield"
    ],
    "trade_external": [
        "export", "import", "trade balance", "tariff", "trade war", "trade gap", 'supply chain', 'disruptions', 'logistics network',
        "exchange rate", "free trade", "devaluation", "trade deficit", "trade surplus", "current account",
        'usd','euro','ringgit','baht','peso','dollar','currency','myr','yen','yuan','php',
        "fx", "forex", "appreciation", "depreciation", "currency peg", "foreign reserves", "international reserves", "spot market", "cross-rate", "undervalued currency",
        "freight", "cargo", "shipping lane", "port congestion", "customs duty", "bill of lading", "maritime trade", "transshipment", "vessel", "container terminal"
    ],
    "fiscal_policy": [
        "government spending", "tax revenue", "budget deficit", "stimulus",
        "public debt", "infrastructure", "austerity", "tax base", "social security", "payroll tax",
        "budget revision", "price control", "targeted subsidy", "targeted subsidies",
        "diesel price", "fuel cost", "govt to allocate", "government allocation",
        "public spending", "subsidy rationalisation", "subsidy rationalization", 'subsidy',
        'economy ministry', 'ministry of economy', 'capital allocation', 'state-backed',
        "allocation", "funded by", "million allocation", "disbursement", "package",'supply'
    ],
    "energy_commodities": [
        "oil", "corn", "wheat", "palm oil", "solar", "wind", "nuclear", "electricity",
        "natural gas", "coal", "renewables", "metals", "price shock", "carbon", "supply disruption"
    ],
    "banking_credit": [
        "lending", "debt", "liquidity stress", "default", "interest rate", "bank",
        "mortgage", "loan", "central bank", "credit", "debit"
    ],
    "central_banks": [
        "bnm", "bank negara", "federal reserve", "fed", "ecb", "european central bank",
        "bank of england", "boe", "bank of japan", "boj", "peoples bank of china", "pboc",
        "reserve bank of australia", "rba", "bank of canada", "boc", "swiss national bank",
        "snb", "reserve bank of india", "rbi", "bank of korea", "banco de mexico", "ubs",
        "bank of brazil", "bcb", "monetary authority of singapore", "mas", "bank of russia", "cbr"
    ],
    "rate_actions": [
        "hike", "cut", "lower", "increase", "decrease", "maintain", "hold", "pause", "tighten", "raise"
    ],
    "advanced_context": [
        r"weather\s+the\s+(?:current\s+)?\w+\s+shock",   # Catches structural resilience (Real_Economic_Activity)
        r"supply\s+shock",                               # Supply chain disruptions (Real_Economic_Activity)
        r"global\s+(?:oil|commodity|energy|market|price)", # International transmission (External_Sector)
        r"war\s+in\s+\w+",                                # Geopolitical macro impact (External_Sector)
        r"external\s+(?:shock|demand|factor|environment)", # Direct external sector mention (External_Sector)
        # Should look for industry indicators coupled with directional market movement
        r"(slowdown|contraction|slump|boom|growth)\s+in\s+.*(sector|industry|real estate|construction)"
    ],
    'fiscal_actions': [
        'urge', 'introduce', 'implement', 'announce', 'adjust', 'reform', 'evaluate','evaluation', 'regard',
        'regulate', 'measure', 'table', 'approve', 'slash', 'hike', 'allocate', 'ensure'
    ],
    'fiscal_instruments': [
        'subsidy', 'tax', 'taxation', 'budget', 'spending', 'expenditure', 'subsidy',
        'tariff', 'levy', 'excise', 'stimulus', 'allocation', 'package', 'supply', 'plan'
    ],
    'labor_actions': [
        'record', 'report', 'register', 'show', 'increase', 'rise'
    ]
}

In [ ]:
# Helper function for fast regex matching
def contains_keywords(text: str, keywords: list[str]) -> bool:
    text_lower = text.lower()
    return any(kw in text_lower for kw in keywords)

PRESENT = 1
ABSENT = 0
ABSTAIN = -1


In [ ]:
# Check if sentence is actually related to economics
@labeling_function(pre=[cache_embeddings])
def lf_is_not_economic_news(x):
    # If it falls way below your baseline anchor similarity, vote ABSENT (0)
    if all(util.cos_sim(x.embedding, anchor) < SEMANTIC_THRESHOLD for anchor in economic_anchor_embeddings):
        return ABSENT
    return ABSTAIN

In [ ]:
import spacy
from spacy.matcher import PhraseMatcher
from spacy.tokens import Token
# Generic keyword factory generator to remove boilerplate code
global nlp
nlp = spacy.load("en_core_web_sm")

def make_keyword_lf(category_name):
    category_keywords = KEYWORDS[category_name]

    matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

    # Pre-lemmatize each keyword, then lowercase
    patterns = [
        nlp.make_doc(" ".join(t.lemma_.lower() for t in nlp(kw)))
        for kw in category_keywords
    ]
    matcher.add(category_name, patterns)

    @nlp_labeling_function()
    def lf_keyword(x):
        # Also lemmatize + lowercase the doc before matching
        lemmatized_doc = nlp.make_doc(" ".join(t.lemma_.lower() for t in x.doc))
        matches = matcher(lemmatized_doc)
        if matches:
            return PRESENT
        return ABSTAIN

    lf_keyword.name = f"lf_{category_name}_nlp_matcher"
    return lf_keyword

# Generic semantic similarity factory generator to remove boilerplate code
def make_semantic_lf(category_name):
  @labeling_function(pre=[cache_embeddings])
  def lf_semantic(x):
    if any(util.cos_sim(x.embedding, anchor) > SEMANTIC_THRESHOLD for anchor in ANCHORS[category_name]):
      return PRESENT
    elif all(util.cos_sim(x.embedding, anchor) < SEMANTIC_THRESHOLD - 0.15 for anchor in ANCHORS[category_name]):
      return ABSENT
    else:
      return ABSTAIN
  lf_semantic.name = f"lf_{category_name}_semantic"
  return lf_semantic

In [ ]:
@labeling_function()
def lf_lemmatized_cost_push(x):
    """
    Tokenizes and lemmatizes the sentence before applying regex matching
    to catch all morphological variants (e.g., pricing -> price, adjustments -> adjustment).
    """
    # 1. Tokenize and Lemmatize the text
    tokens = word_tokenize(x.text.lower())
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens]

    # Reconstruct the text into a clean string of base words
    processed_text = " ".join(lemmatized_tokens)

    # 2. Direct Fallback Keywords (now in base form)
    if re.search(r"(inflation|inflationary|cost of living|purchasing power)", processed_text):
        return PRESENT

    # 3. Base-word Component Patterns
    # Notice we use the root form here (e.g., 'utility' instead of 'utilities')
    energy_inputs = r"(diesel|fuel|petrol|oil|energy|utility|electricity|pump price)"
    price_adjustments = r"(adjustment|hike|spike|rise|increase|soaring|escalate|tariff|price|charge)"
    systemic_impact = r"(consumer|business|enterprise|industry|household|supply chain|economic effect|burden|margin)"

    # NEW: Macro economic shock indicators for classic cost-push inflation drivers
    macro_shocks = r"(supply shock|commodity shock|import cost|raw material shortage|wage spiral)"

    # 4. Multi-component Matching
    has_energy = re.search(energy_inputs, processed_text)
    has_adjustment = re.search(price_adjustments, processed_text)
    has_impact = re.search(systemic_impact, processed_text)

    # NEW: Check if the text explicitly cites a macro supply shock alongside pricing terms
    has_macro_shock = re.search(macro_shocks, processed_text)

    # Trigger if it hits the original multi-component energy/utility impact matrix
    if has_energy and has_adjustment and has_impact:
        return PRESENT

    # FIX TRIGGER: Trigger if a explicit supply/macro shock is directly linked
    # to a price adjustment/term (e.g., "supply shock fits into domestic price")
    if has_macro_shock and has_adjustment:
        return PRESENT

    return ABSTAIN

In [ ]:
# Complex Conditional Rule Tuning
@labeling_function()
def lf_monetary_policy_inflation_exclusion(x):
    """
    Disambiguates institutional monetary policy from household cost-of-living metrics.
    """
    has_inflation = contains_keywords(x.text, KEYWORDS['inflation'])
    has_monetary = contains_keywords(x.text, KEYWORDS['monetary_policy'])
    has_consumer = contains_keywords(x.text, KEYWORDS['consumer_activity'])

    if has_inflation and has_consumer:
        return ABSTAIN
    return PRESENT if (has_monetary and has_inflation) else ABSTAIN

# Checks for central banks
@nlp_labeling_function()
def lf_central_banks_nlp(x):
    """
    Isolates Central Bank institution mentions.
    Target Aspects: Monetary_Financial, Fiscal_Government
    """
    for ent in x.doc.ents:
        if ent.label_ == "ORG" and ent.text.lower() in KEYWORDS['central_banks']:
            return PRESENT
    return ABSTAIN

# Checks if there is a mention of a rate change
@nlp_labeling_function()
def lf_rate_actions_nlp(x):
    """
    Isolates precise grammar structures where interest rates are being altered/held.
    Target Aspects: Monetary_Financial, Inflation_Prices, Fiscal_Government
    """
    for token in x.doc:
        if token.text.lower() == "rate" and token.dep_ in ("dobj", "nsubj", "pobj"):
            if token.head.lemma_ in KEYWORDS['rate_actions']:
                return PRESENT
    return ABSTAIN

@nlp_labeling_function()
def lf_labor_market_action_nlp(x):
    """
    Isolates corporate or market-driven labor movements (e.g., "Companies lay off..."),
    state-driven worker protection adjustments, and labor statistics shifts (e.g., population headcount changes).
    Target Aspects: Labor_Consumption
    """
    # 1. Loop through ALL tokens
    for token in x.doc:

        # 2. Focus on subjects, passive subjects, and prepositional objects
        if token.dep_ in ("nsubj", "nsubjpass", "pobj"):

            # Core Check: If the token is a labor market element
            if token.lemma_ in KEYWORDS['labor_market']:

                # Check the direct governing head first
                if token.head.lemma_ in KEYWORDS['labor_actions']:
                    return PRESENT

                # FALLBACK FIX: Climb the dependency tree (Ancestors)
                # This handles statistical layouts (e.g., "number [of] workers [recorded] a [rise]")
                for i, ancestor in enumerate(token.ancestors):
                    if i >= 3: # Keep it efficient and bounded to 3 levels
                        break
                    if ancestor.lemma_ in KEYWORDS['labor_actions'] or ancestor.lemma_ in ("rise", "increase", "drop", "growth", "decline", "fall"):
                        return PRESENT

            # Alternative Check: If the governor is a labor term, look at the token itself
            elif token.head.lemma_ in KEYWORDS['labor_market']:
                if token.lemma_ in KEYWORDS['labor_actions'] or token.lemma_ in ("rise", "increase", "drop", "growth", "decline", "fall"):
                    return PRESENT

    return ABSTAIN

In [ ]:
@labeling_function()
def lf_central_bank_fiscal_exclusion(x):
    """
    Abstains or rejects if a central bank is speaking without clear fiscal context,
    preventing institutional bleed.
    """
    has_central_bank = contains_keywords(x.text, KEYWORDS['central_banks'])
    has_fiscal_keywords = contains_keywords(x.text, KEYWORDS['fiscal_policy'])

    # If a central bank is talking, but there are no government spending/tax keywords,
    # actively force an ABSTAIN (or return 0 if you want it to vote ABSENT)
    if has_central_bank and not has_fiscal_keywords:
        return ABSENT

    return ABSTAIN

@labeling_function()
def lf_fiscal_sector_clash_exclusion(x):
    """
    If a sentence is overwhelmingly tracking heavy monetary or external macro metrics
    WITHOUT fiscal keywords, explicitly vote 0 (Absent) instead of abstaining.
    """
    has_monetary = contains_keywords(x.text, KEYWORDS['monetary_policy'])
    has_external = contains_keywords(x.text, KEYWORDS['trade_external']) or contains_keywords(x.text, KEYWORDS['energy_commodities'])
    has_fiscal = contains_keywords(x.text, KEYWORDS['fiscal_policy'])

    # If it's heavily monetary/external but lacks fiscal words, vote ABSENT (0)
    if (has_monetary or has_external) and not has_fiscal:
        return ABSENT

    return ABSTAIN

@labeling_function()
def lf_monetary_exclusion_clash(x):
    """
    Votes ABSENT (0) if the sentence is heavily focused on pure fiscal budgets
    or corporate labor actions without any monetary indicator.
    """
    has_monetary = contains_keywords(x.text, KEYWORDS['monetary_policy']) or contains_keywords(x.text, KEYWORDS['financial_markets'])
    has_fiscal_heavy = contains_keywords(x.text, ["budget deficit", "austerity", "tax base", "tax revenue"])
    has_labor_heavy = contains_keywords(x.text, ["layoffs", "unemployment", "hiring", "minimum wage"])

    if (has_fiscal_heavy or has_labor_heavy) and not has_monetary:
        return ABSENT
    return ABSTAIN

@labeling_function()
def lf_inflation_exclusion_clash(x):
    """
    Votes ABSENT (0) if text is strictly about equity market adjustments, corporate
    automation, or trade treaties without direct price pressure vocabulary.
    """
    has_inflation = contains_keywords(x.text, KEYWORDS['inflation'])
    has_market_heavy = contains_keywords(x.text, ["bursa", "equity", "stock index", "yield curve"])
    has_corp_heavy = contains_keywords(x.text, ["automation", "property rights", "red tape", "free trade"])

    if (has_market_heavy or has_corp_heavy) and not has_inflation:
        return ABSENT
    return ABSTAIN

@labeling_function()
def lf_activity_exclusion_clash(x):
    """
    Votes ABSENT (0) if the text is confined to central bank rate holding syntax
    or consumer debt metrics with no production or growth keywords.
    """
    has_activity = contains_keywords(x.text, KEYWORDS['economic_growth']) or contains_keywords(x.text, KEYWORDS['business_activity'])
    has_rate_heavy = contains_keywords(x.text, ["rate cut", "rate hike", "opr", "overnight policy rate"])
    has_consumer_debt = contains_keywords(x.text, ["credit card usage", "household debt"])

    if (has_rate_heavy or has_consumer_debt) and not has_activity:
        return ABSENT
    return ABSTAIN

@labeling_function()
def lf_labor_exclusion_clash(x):
    """
    Votes ABSENT (0) if the text details cross-border energy/commodity markets
    or sovereign public debt dynamics with no mention of workers or consumers.
    """
    has_labor_cons = contains_keywords(x.text, KEYWORDS['labor_market']) or contains_keywords(x.text, KEYWORDS['consumer_activity'])
    has_energy_heavy = contains_keywords(x.text, KEYWORDS['energy_commodities'])
    has_sovereign_debt = contains_keywords(x.text, ["public debt", "budget deficit", "sovereign bond"])

    if (has_energy_heavy or has_sovereign_debt) and not has_labor_cons:
        return ABSENT
    return ABSTAIN

@labeling_function()
def lf_fiscal_exclusion_clash(x):
    """
    Votes ABSENT (0) if a sentence is overwhelmingly focused on monetary policy
    mechanisms or cross-border trade balances without any legislative government anchors.
    """
    has_fiscal = contains_keywords(x.text, KEYWORDS['fiscal_policy'])
    has_monetary_heavy = contains_keywords(x.text, KEYWORDS['monetary_policy']) or contains_keywords(x.text, KEYWORDS['central_banks'])
    has_trade_heavy = contains_keywords(x.text, KEYWORDS['trade_external'])

    if (has_monetary_heavy or has_trade_heavy) and not has_fiscal:
        return ABSENT
    return ABSTAIN

@labeling_function()
def lf_external_exclusion_clash(x):
    """
    Votes ABSENT (0) if the text handles hyper-localized economic variables
    like domestic minimum wages, national retail sales, or localized infrastructure.
    """
    has_external = contains_keywords(x.text, KEYWORDS['trade_external']) or contains_keywords(x.text, KEYWORDS['energy_commodities'])
    has_local_heavy = contains_keywords(x.text, ["minimum wage", "retail sales", "payroll tax", "infrastructure"])

    if has_local_heavy and not has_external:
        return ABSENT
    return ABSTAIN

@labeling_function()
def lf_deny_fiscal_if_pure_labor(x):
    labor_indicators = {"employers", "wages", "dismissals", "workers", "leave", "strike"}
    fiscal_indicators = {"government", "tax", "budget", "subsidy", "ministry", "fiscal"}

    text_lower = x.text.lower()

    # If it is explicitly about workplace rights and mentions NO government apparatus
    if any(word in text_lower for word in labor_indicators) and not any(word in text_lower for word in fiscal_indicators):
        return ABSENT  # Explicitly vote ABSENT to break the passive voting pattern
    return ABSTAIN

In [ ]:
# Pre-compile the regex pattern outside the function loop for maximum speed
COMPILED_CONTEXT_REGEX = re.compile("|".join(KEYWORDS["advanced_context"]), re.IGNORECASE)

@labeling_function()
def lf_advanced_context_regex(x):
    """
    Scans for complex structural phrases representing external sector impacts
    and real economic activity shocks in a single high-speed pass.
    """
    return PRESENT if COMPILED_CONTEXT_REGEX.search(x.text) else ABSTAIN

In [ ]:
@labeling_function(pre=[cache_embeddings])
def lf_fiscal_policy_semantic_true(x):
    # Fires PRESENT (1) only if similarity is very high
    if any(util.cos_sim(x.embedding, anchor) > 0.4 for anchor in ANCHORS['fiscal_policy']):
        return PRESENT
    return ABSTAIN

@labeling_function(pre=[cache_embeddings])
def lf_fiscal_policy_semantic_false(x):
    # Explicitly flags ABSENT (0) if it matches non-fiscal anchors
    if all(util.cos_sim(x.embedding, anchor) > 0.2 for anchor in ANCHORS['fiscal_policy']):
        return ABSENT
    return ABSTAIN

In [ ]:
@nlp_labeling_function()
def lf_fiscal_actions_nlp(x):
    """
    Isolates precise grammar structures where fiscal instruments (taxes, subsidies, budgets, packages)
    are being acted upon by a governing verb, or where government policy is explicitly directed.
    Target Aspects: Fiscal_Government
    """
    for token in x.doc:
        # Check if the current noun is a known fiscal instrument
        if token.lemma_ in KEYWORDS['fiscal_instruments']:

            # FIX: Added 'ccomp' to handle reported-speech news layout patterns
            if token.dep_ in ("dobj", "nsubj", "nsubjpass", "pobj", "ccomp"):

                # Check the direct governing verb
                if token.head.lemma_ in KEYWORDS['fiscal_actions']:
                    return PRESENT

                # FALLBACK FIX 1: Look up the tree (Ancestors)
                # Handles nested prepositional phrases and split hyphens (e.g., "re-evaluation -> of -> subsidies")
                # Restricts search to a maximum depth of 3 to avoid false positive triggers elsewhere in long sentences
                for i, ancestor in enumerate(token.ancestors):
                    if i >= 3:
                        break
                    if ancestor.lemma_ in KEYWORDS['fiscal_actions']:
                        return PRESENT

                # FALLBACK FIX 2: Check if the token's children/advcl contain the action verb
                # (Handles when a currency number splits the token dependency tree)
                for child in token.children:
                    if child.lemma_ in KEYWORDS['fiscal_actions']:
                        return PRESENT

        # Rule 2: Expanded to include "minister" and "package"
        if token.lemma_ in ("government", "ministry", "policy", "measure", "minister", "package"):
            if token.dep_ in ("dobj", "nsubj", "nsubjpass", "pobj", "ccomp") and token.head.lemma_ in KEYWORDS['fiscal_actions']:
                return PRESENT

            # Check ancestors for Rule 2 as well to capture multi-word structural policy mentions
            for i, ancestor in enumerate(token.ancestors):
                if i >= 3:
                    break
                if token.dep_ in ("dobj", "nsubj", "nsubjpass", "pobj", "ccomp") and ancestor.lemma_ in KEYWORDS['fiscal_actions']:
                    return PRESENT

    return ABSTAIN

In [ ]:
# When extracting your final training labels from the matrix:
def filter_high_confidence_solo_votes(label_matrix, probabilities, threshold=0.5):
    final_labels = []

    for row_votes, prob in zip(label_matrix, probabilities):
        # Count how many active (non-abstain) votes are present
        # (Assuming ABSTAIN is coded as -1 or 0 depending on your setup)
        active_votes = sum(1 for vote in row_votes if vote != -1)

        # Guardrail: If only 1 LF voted, don't trust a runaway high probability
        if active_votes == 1 and prob > 0.90:
            final_labels.append(-1) # Force to ABSTAIN/Ignore for training
        else:
            final_labels.append(1 if prob >= threshold else 0)

    return final_labels

In [ ]:
# Instantiating the merged, simplified keyword and semantic similarity LFs
lf_monetary_policy_keywords = make_keyword_lf("monetary_policy")
lf_monetary_policy_semantic = make_semantic_lf("monetary_policy")
lf_inflation_keywords = make_keyword_lf("inflation")
lf_inflation_semantic = make_semantic_lf("inflation")
lf_economic_growth_keywords = make_keyword_lf("economic_growth")
lf_economic_growth_semantic = make_semantic_lf("economic_growth")
lf_labor_market_keywords = make_keyword_lf("labor_market")
lf_labor_market_semantic = make_semantic_lf("labor_market")
lf_consumer_activity_keywords = make_keyword_lf("consumer_activity")
lf_consumer_activity_semantic = make_semantic_lf("consumer_activity")
lf_business_activity_keywords = make_keyword_lf("business_activity")
lf_business_activity_semantic = make_semantic_lf("business_activity")
lf_financial_markets_keywords = make_keyword_lf("financial_markets")
lf_financial_markets_semantic = make_semantic_lf("financial_markets")
lf_trade_external_keywords = make_keyword_lf("trade_external")
lf_trade_external_semantic = make_semantic_lf("trade_external")
lf_fiscal_policy_keywords = make_keyword_lf("fiscal_policy")
lf_fiscal_policy_semantic = make_semantic_lf("fiscal_policy")
lf_energy_commodities_keywords = make_keyword_lf("energy_commodities")
lf_energy_commodities_semantic = make_semantic_lf("energy_commodities")
lf_banking_credit_keywords = make_keyword_lf("banking_credit")
lf_banking_credit_semantic = make_semantic_lf("banking_credit")
lf_central_banks_keywords = make_keyword_lf("central_banks")
lf_rate_actions = make_keyword_lf("rate_actions")

In [ ]:
ASPECTS = [ # 6 MAIN ASPECTS TO LOOK FOR IN TEXT
    "Monetary_Financial",
    "Inflation_Prices",
    "Real_Economic_Activity",
    "Labor_Consumption",
    "Fiscal_Government",
    "External_Sector"
]

monetary_lfs = [
    lf_is_not_economic_news,
    lf_monetary_policy_keywords,
    lf_monetary_policy_semantic,
    lf_financial_markets_keywords,
    lf_financial_markets_semantic,
    lf_banking_credit_keywords,
    lf_banking_credit_semantic,
    lf_central_banks_nlp,
    lf_rate_actions_nlp,
    lf_monetary_policy_inflation_exclusion,
    lf_monetary_exclusion_clash
]

inflation_lfs = [
    lf_is_not_economic_news,
    lf_inflation_keywords,
    lf_inflation_semantic,
    lf_monetary_policy_inflation_exclusion,
    lf_rate_actions_nlp,
    lf_inflation_exclusion_clash,
    lf_lemmatized_cost_push
]

activity_lfs = [
    lf_is_not_economic_news,
    lf_economic_growth_keywords,
    lf_economic_growth_semantic,
    lf_business_activity_keywords,
    lf_business_activity_semantic,
    lf_trade_external_keywords,
    lf_activity_exclusion_clash,
    lf_advanced_context_regex  # Highly critical here to catch physical "supply shocks"
]

labor_lfs = [
    lf_is_not_economic_news,
    lf_labor_market_keywords,
    lf_labor_market_semantic,
    lf_labor_exclusion_clash,
    lf_consumer_activity_keywords,
    lf_labor_market_action_nlp    # Catches structural labor-actions via dependency parsing
]

fiscal_lfs = [
    lf_is_not_economic_news,
    lf_fiscal_policy_keywords,
    lf_fiscal_policy_semantic,
    lf_central_bank_fiscal_exclusion,
    lf_deny_fiscal_if_pure_labor,
    lf_fiscal_actions_nlp
]

external_lfs = [
    lf_is_not_economic_news,
    lf_trade_external_keywords,
    lf_trade_external_semantic,
    lf_energy_commodities_keywords,
    lf_energy_commodities_semantic,
    lf_external_exclusion_clash,
    lf_advanced_context_regex  # Highly critical here to catch "global oil prices" and geopolitical boundaries
]

# 2. Final Aspect Mapping Dictionary
aspect_lf_groups = {
    "Monetary_Financial": monetary_lfs,
    "Inflation_Prices": inflation_lfs,
    "Real_Economic_Activity": activity_lfs,
    "Labor_Consumption": labor_lfs,
    "Fiscal_Government": fiscal_lfs,
    "External_Sector": external_lfs
}

In [ ]:

nltk.download('punkt_tab', quiet=True)

# ── 1. Load the CSV ──────────────────────────────────────────────────────────
#df = pd.read_csv("malaysian_economic_news_1000.csv")   # ← update path as needed


True

In [ ]:
# ── 2. Explode each article into one row per sentence ────────────────────────
def explode_to_sentences(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for article_id, row in df.iterrows():
        title = row.get("title", "")
        text  = row.get("text", "")

        # headline as its own sentence row
        if pd.notna(title) and str(title).strip():
            rows.append({
                "article_id":   article_id,
                "title":        title,
                "sentence_idx": -1,           # -1 flags it as the headline
                "text":         str(title).strip(),
                "is_headline":  True,
            })

        # body sentences
        if pd.notna(text) and str(text).strip():
            sentences = sent_tokenize(str(text))
            for i, sent in enumerate(sentences):
                rows.append({
                    "article_id":   article_id,
                    "title":        title,
                    "sentence_idx": i,
                    "text":         sent.strip(),
                    "is_headline":  False,
                })

    return pd.DataFrame(rows)

In [ ]:
sentence_df = explode_to_sentences(df)
print(f"\nTotal sentences: {len(sentence_df)}")
print(f"Total articles:  {sentence_df['article_id'].nunique()}")


Total sentences: 3475
Total articles:  182


In [ ]:
sample_df = sentence_df.sample(n=500, random_state=42)

In [ ]:
sample_df.to_csv("sample_df.csv", index=False)

In [ ]:
# We'll store the LF matrices for each aspect
L_matrices = {}

for aspect, lf_list in aspect_lf_groups.items():
  applier = PandasLFApplier(lf_list)
  L_train = applier.apply(df)
  L_matrices[aspect] = L_train


100%|██████████| 47/47 [00:00<00:00, 329.80it/s]


In [ ]:
# Train a LabelModel per aspect
label_models = {}
preds_proba = {}  # shape: (num_samples, 1) probability of class 1

for aspect, L in L_matrices.items():
    label_model = LabelModel(cardinality=2, verbose=False)
    label_model.fit(L, n_epochs=20, seed=42)
    label_models[aspect] = label_model
    # Get probabilistic predictions for class 1
    preds_proba[aspect] = label_model.predict_proba(L)[:, 1]  # probability of presence

100%|██████████| 20/20 [00:00<00:00, 852.45epoch/s]


In [ ]:
for aspect, L in L_matrices.items():
    # 1. Add precise class balance priors if you know roughly how much data is fiscal
    # 2. Add L2 regularization (prec_init) to prevent the model from tanking the weights of low-coverage LFs
    label_model = LabelModel(
        cardinality=2,
        verbose=True
    )

    # Use fit with a specified optimizer learning rate and a higher penalty on structural assumptions
    label_model.fit(
        L,
        n_epochs=100,
        seed=42,
        lr=0.01,              # Lower learning rate avoids aggressive weight drops
        l2=0.1                # Adds slight regularization to stabilize the matrix
    )

    label_models[aspect] = label_model
    preds_proba[aspect] = label_model.predict_proba(L)[:, 1]

100%|██████████| 100/100 [00:00<00:00, 949.42epoch/s]


In [ ]:
import numpy as np

for aspect, L in L_matrices.items():
    # 1. Initialize the LabelModel
    label_model = LabelModel(
        cardinality=2,
        verbose=True
    )

    # 2. Fit with a specified optimizer learning rate and regularization penalty
    label_model.fit(
    L,
    n_epochs=100,
    seed=42,
    lr=0.01,
    l2=0.1,
)

    label_models[aspect] = label_model

    # 3. Get raw continuous probability scores for the PRESENT class (col 1)
    raw_probs = label_model.predict_proba(L)[:, 1]

    # --------------------------------------------------------------------------
    # GUARDRAIL: Identify and filter out runaway high-confidence solo votes
    # --------------------------------------------------------------------------
    # Count how many labeling functions voted (anything that isn't an ABSTAIN / -1)
    active_vote_counts = (L != -1).sum(axis=1)

    # Create a mask for rows where exactly ONE labeling function voted
    is_solo_vote = (active_vote_counts == 1)

    # Create a mask for rows where the model generated a runaway high probability
    is_runaway_confidence = (raw_probs > 0.80)

    # Combine masks: True if it's a dangerous solo voter with runaway confidence
    should_filter = is_solo_vote & is_runaway_confidence

    # Apply guardrail: Force dangerous rows to 0.0 probability (or map to a specific fallback index)
    sanitized_probs = np.where(should_filter, 0.0, raw_probs)

    # Store the sanitized probabilities for your final downstream extraction
    preds_proba[aspect] = sanitized_probs

100%|██████████| 100/100 [00:00<00:00, 939.71epoch/s]


In [ ]:
for aspect, lf_list in aspect_lf_groups.items():
  print(f'\nASPECT: {aspect}')
  print(LFAnalysis(L=L_matrices[aspect], lfs=lf_list).lf_summary())



ASPECT: Monetary_Financial
                                         j Polarity  Coverage  Overlaps  \
lf_is_not_economic_news                  0      [0]  0.191489  0.191489   
lf_monetary_policy_nlp_matcher           1      [1]  0.148936  0.148936   
lf_monetary_policy_semantic              2   [0, 1]  0.553191  0.531915   
lf_financial_markets_nlp_matcher         3      [1]  0.063830  0.042553   
lf_financial_markets_semantic            4   [0, 1]  0.446809  0.382979   
lf_banking_credit_nlp_matcher            5      [1]  0.085106  0.085106   
lf_banking_credit_semantic               6      [0]  0.617021  0.489362   
lf_central_banks_nlp                     7      [1]  0.042553  0.042553   
lf_rate_actions_nlp                      8       []  0.000000  0.000000   
lf_monetary_policy_inflation_exclusion   9      [1]  0.085106  0.085106   
lf_monetary_exclusion_clash             10      [0]  0.085106  0.063830   

                                        Conflicts  
lf_is_not_economic_

In [ ]:
# Define the order you want (same as aspect_lf_groups)
aspect_order = [
    "Monetary_Financial",
    "Inflation_Prices",
    "Real_Economic_Activity",
    "Labor_Consumption",
    "Fiscal_Government",
    "External_Sector"
]

# Stack probabilities into an (n_samples, 6) array
probs_matrix = np.column_stack([preds_proba[aspect] for aspect in aspect_order])

# Or as a tidy DataFrame (each row is a sample, each column an aspect)
probs_df = pd.DataFrame(probs_matrix, columns=aspect_order)

# If you later need crisp 0/1 presence labels, threshold (e.g., at 0.5):
#presence_matrix = (probs_matrix >= 0.5).astype(int)
#presence_df = pd.DataFrame(presence_matrix, columns=aspect_order)

In [ ]:

# ── 1. Concatenate ────────────────────────────────────────────────────────────
# probs_df columns are your label model probability outputs
#combined_df = pd.concat([sample_df.reset_index(drop=True), probs_df.reset_index(drop=True)], axis=1)
combined_df = pd.concat([df.reset_index(drop=True), probs_df.reset_index(drop=True)], axis=1)

print(combined_df.head())
print(f"\nShape: {combined_df.shape}")

# ── 2. Check for predictions significantly close to 0 or 1 ───────────────────
THRESHOLD = 0.15  # "close to 0" = < 0.15, "close to 1" = > 0.85

label_cols = probs_df.columns.tolist()

# Boolean mask: any label column is near 0 or 1
near_boundary = combined_df[label_cols].apply(
    lambda col: (col < THRESHOLD) | (col > (1 - THRESHOLD))
)

# Rows where AT LEAST ONE label is near a boundary
flagged = combined_df[near_boundary.any(axis=1)].copy()
flagged["flagged_labels"] = near_boundary[near_boundary.any(axis=1)].apply(
    lambda row: [label_cols[i] for i, v in enumerate(row) if v], axis=1
)

print(f"\nFlagged rows (any label near 0 or 1): {len(flagged)} / {len(combined_df)}")
print(flagged[["article_id", "text", "flagged_labels"] + label_cols].head(10))

# ── 3. Summary: per-label breakdown ──────────────────────────────────────────
print("\n── Per-label flagged counts ──")
for col in label_cols:
    n_near_zero = (combined_df[col] < THRESHOLD).sum()
    n_near_one  = (combined_df[col] > (1 - THRESHOLD)).sum()
    print(f"  {col:30s}  near 0: {n_near_zero:4d}  |  near 1: {n_near_one:4d}")

                               aspects  article_id  \
0                                 None         164   
1                                 None           3   
2                      External_Sector          97   
3                      External_Sector          65   
4  External_Sector + Fiscal_Government         159   

                                               title  sentence_idx  \
0         Where experience meets academic excellence            14   
1  ECB urges banks to brace quickly for AI-assist...             8   
2  China's durian boom is ripening into a regiona...            21   
3  Johor eyeing 12 million visitors, RM42.48bil i...             7   
4              Cautious optimism on growth prospects             4   

                                                text  Monetary_Financial  \
0  “I needed a university that was recognised, ap...            0.018252   
1  (Reporting by Ludwig Burger and Reinhard Becke...            0.106822   
2  As Southeast Asia's mai

In [ ]:
# ── Filter by aspect ──────────────────────────────────────────────────────────
ASPECT = "Fiscal_Government"   # ← change this to any label col
DIRECTION = "near_1"            # ← "near_1", "near_0", or "both"

if DIRECTION == "near_1":
    mask = combined_df[ASPECT] > (1 - THRESHOLD)
elif DIRECTION == "near_0":
    mask = combined_df[ASPECT] < THRESHOLD
else:  # both
    mask = (combined_df[ASPECT] > (1 - THRESHOLD)) | (combined_df[ASPECT] < THRESHOLD)

aspect_flagged = combined_df[mask][["article_id", "text", ASPECT]]

print(f"── {ASPECT} | {DIRECTION} | {len(aspect_flagged)} rows ──")
for _, row in aspect_flagged.iterrows():
    print(f"\n [{row["article_id"]}] [{row[ASPECT]:.4f}]  {row['text'][:120]}")

── Fiscal_Government | near_1 | 15 rows ──

 [174] [0.9819]  “As higher energy costs pass through supply chains worldwide, a broader range of Singapore’s import costs will increase.

 [81] [0.9819]  This initiative will be funded by TalentCorp with an allocation of RM10mil,' he said.

 [91] [0.9819]  Mohideen added that a portion of these savings could be channelled into food subsidies for the poorest groups to help cu

 [141] [0.9991]  He urged the government to recognise the wider economic effects of diesel price adjustments and to introduce targeted me

 [81] [0.9991]  KUALA LUMPUR: An economic resilience package worth more than RM710mil will be implemented to ensure the country is prepa

 [81] [0.9819]  “To diversify skills and strengthen social security protection for gig workers, the government has channelled RM20mil to

 [59] [0.9991]  Fernandes said that to achieve and maintain growth, the country must make difficult decisions, especially regarding the 

 [28] [0.9991]  PM to a

In [ ]:
view_df = combined_df[combined_df["article_id"].isin([150, 16])][['article_id',"text", "Monetary_Financial", "Inflation_Prices", "Real_Economic_Activity",
    "Labor_Consumption", "Fiscal_Government", "External_Sector"]]

In [ ]:
#view_df.iloc[1]
view_df.head(10)

,article_id,text,Monetary_Financial,Inflation_Prices,Real_Economic_Activity,Labor_Consumption,Fiscal_Government,External_Sector
28,16,Teves said the market would be watching for ke...,0.297402,0.999996,0.962703,0.297374,0.500000,0.500000
36,150,The ringgit gained against the Thai baht to 12...,0.282856,0.500000,0.040400,0.297374,0.500000,0.982779
37,16,"Gold seen bullish, to hit US$5,600 by year-end...",0.027321,0.302371,0.000834,0.154389,0.000907,0.008003


In [ ]:
chosen = view_df.iloc[2]
print(f": [text]: "+chosen.text)
for aspect, lf_list in aspect_lf_groups.items():
    print(f"\n  {aspect:20s} {chosen[aspect]}")
    print(f"  {'-' * 40}")
    for lf in lf_list:
        result = lf(chosen)
        label  = "PRESENT" if result == 1 else "ABSTAIN" if result == -1 else "ABSENT"
        print(f"    {lf.name:40s}  →  {label}")

print("\n" + "=" * 60)

: [text]: Gold seen bullish, to hit US$5,600 by year-end - UBS

  Monetary_Financial   0.02732145220626158
  ----------------------------------------
    lf_is_not_economic_news                   →  ABSENT
    lf_monetary_policy_nlp_matcher            →  ABSTAIN
    lf_monetary_policy_semantic               →  ABSENT
    lf_financial_markets_nlp_matcher          →  ABSTAIN
    lf_financial_markets_semantic             →  ABSTAIN
    lf_banking_credit_nlp_matcher             →  ABSTAIN
    lf_banking_credit_semantic                →  ABSENT
    lf_central_banks_nlp                      →  ABSTAIN
    lf_rate_actions_nlp                       →  ABSTAIN
    lf_monetary_policy_inflation_exclusion    →  ABSTAIN
    lf_monetary_exclusion_clash               →  ABSTAIN

  Inflation_Prices     0.3023705125951956
  ----------------------------------------
    lf_is_not_economic_news                   →  ABSENT
    lf_inflation_nlp_matcher                  →  ABSTAIN
    lf_inflation_semantic  

In [ ]:
embedding = embedder.encode(chosen.text, convert_to_tensor=True)
print(f'Similarity between:\n{chosen.text} and:\n')
for anchor in ECONOMIC_ANCHORS:
    anchor_embedding = embedder.encode(anchor, convert_to_tensor=True)
    similarity = util.cos_sim(embedding,anchor_embedding)
    print(f"{anchor}\n{similarity.item():.4f}")



Similarity between:
Gold seen bullish, to hit US$5,600 by year-end - UBS and:

Gold seen bearish, according to global investor sentiments as reported by central bank
0.5196
A high-ranking government minister faces official questioning and institutional scrutiny regarding state-backed investment deals, public venture funds, or corporate partnerships with multinational tech firms.
0.0264
Government entities and ministries face regulatory oversight and institutional scrutiny regarding public capital allocations for strategic foreign direct investments, sovereign tech venture funding, and semiconductor supply chain partnerships.
0.0093
Analysts warned that prolonged inflation could weigh further on economic recovery prospects.
0.1271
Sovereign nations utilizing international emergency credit lines must settle outstanding central banking obligations and institutional stabilization loans to restore external financial credibility.
0.0525
Financial media outlets and international institutions 

In [ ]:
lf_fiscal_policy_keywords(chosen)

-1

In [ ]:
KEYWORDS['fiscal_policy']

['government spending',
 'tax revenue',
 'budget deficit',
 'stimulus',
 'public debt',
 'infrastructure',
 'austerity',
 'tax base',
 'social security',
 'payroll tax',
 'budget revision',
 'price control',
 'targeted subsidy',
 'targeted subsidies',
 'diesel price',
 'fuel cost',
 'govt to allocate',
 'government allocation',
 'public spending',
 'subsidy rationalisation',
 'subsidy rationalization',
 'subsidy',
 'economy ministry',
 'ministry of economy',
 'capital allocation',
 'state-backed',
 'allocation',
 'funded by',
 'million allocation',
 'disbursement',
 'package',
 'supply']

In [ ]:
lf_fiscal_policy_keywords(chosen)

1

In [ ]:
results = []
for idx, row in df.iterrows():
    result = lf_fiscal_policy_semantic_true(row)
    if result == 1:  # ABSENT
        results.append({"index": idx, "text": row.text,"article_id": row.article_id})

print(f"ABSENT rows: {len(results)} / {len(sample_df)}\n")
for r in results:
    print(f"  [{r['article_id']}]  {r['text'][:120]}")

ABSENT rows: 5 / 500

  [141]  He urged the government to recognise the wider economic effects of diesel price adjustments and to introduce targeted me
  [103]  Faced with the risk of sovereign default, the country turned to a massive bailout led by the International Monetary Fund
  [103]  Faced with the risk of sovereign default, the country turned to a massive bailout led by the International Monetary Fund
  [54]  At the same time, there is also a valid concern on global growth prospects, which may necessitate easing monetary policy
  [149]  Private investment is projected to stay robust, driven by infrastructure rollouts, data centre developments, and continu


In [ ]:
# 1. Parse the specific sentence with your loaded spacy model
doc = nlp(chosen.text)
print(f"{'TOKEN':<15} | {'LEMMA':<15} | {'DEP TAG':<10} | {'HEAD VERB (LEMMA)':<18}")
print("-" * 65)

for token in doc:
    print(f"{token.text:<15} | {token.lemma_:<15} | {token.dep_:<10} | {token.head.lemma_:<18}")

TOKEN           | LEMMA           | DEP TAG    | HEAD VERB (LEMMA) 
-----------------------------------------------------------------
Fernandes       | Fernandes       | nsubj      | say               
said            | say             | ROOT       | say               
that            | that            | mark       | make              
to              | to              | aux        | achieve           
achieve         | achieve         | advcl      | make              
and             | and             | cc         | achieve           
maintain        | maintain        | conj       | achieve           
growth          | growth          | dobj       | maintain          
,               | ,               | punct      | make              
the             | the             | det        | country           
country         | country         | nsubj      | make              
must            | must            | aux        | make              
make            | make            | ccomp      | s

In [ ]:
# Check how spaCy tokenizes your keyword vs the text
print([token.lemma_ for token in nlp("economy ministry")])
print([token.lemma_ for token in nlp("Economy Ministry")])

# Check what the matcher actually sees in your doc
doc = nlp(chosen.text)
print([(token.text, token.lemma_) for token in doc])

['economy', 'ministry']
['Economy', 'Ministry']
[('Rafizi', 'Rafizi'), ('is', 'be'), ('being', 'be'), ('questioned', 'question'), ('over', 'over'), ('allegations', 'allegation'), ('linked', 'link'), ('to', 'to'), ('a', 'a'), ('semiconductor', 'semiconductor'), ('investment', 'investment'), ('deal', 'deal'), ('involving', 'involve'), ('the', 'the'), ('Economy', 'Economy'), ('Ministry', 'Ministry'), ('and', 'and'), ('Arm', 'Arm'), ('Holdings', 'Holdings'), ('.', '.')]


In [ ]:
matcher_test = PhraseMatcher(nlp.vocab, attr="LEMMA")
matcher_test.add("test", [nlp("economy ministry")])
print(matcher_test(nlp(chosen.text)))

[]


In [ ]:
from spacy.tokens import Token
Token.set_extension("lower_lemma", getter=lambda t: t.lemma_.lower(), force=True)
lf_fiscal_policy_keywords = make_keyword_lf("fiscal_policy")
doc = nlp(chosen.text)
print([(token.text, token.lemma_, token._.lower_lemma) for token in doc])

[('Rafizi', 'Rafizi', 'rafizi'), ('is', 'be', 'be'), ('being', 'be', 'be'), ('questioned', 'question', 'question'), ('over', 'over', 'over'), ('allegations', 'allegation', 'allegation'), ('linked', 'link', 'link'), ('to', 'to', 'to'), ('a', 'a', 'a'), ('semiconductor', 'semiconductor', 'semiconductor'), ('investment', 'investment', 'investment'), ('deal', 'deal', 'deal'), ('involving', 'involve', 'involve'), ('the', 'the', 'the'), ('Economy', 'Economy', 'economy'), ('Ministry', 'Ministry', 'ministry'), ('and', 'and', 'and'), ('Arm', 'Arm', 'arm'), ('Holdings', 'Holdings', 'holdings'), ('.', '.', '.')]


In [ ]:
pattern_doc = nlp("economy ministry")
print([(token.text, token._.lower_lemma) for token in pattern_doc])

AttributeError: [E046] Can't retrieve unregistered extension attribute 'lower_lemma'. Did you forget to call the `set_extension` method?